# Token-level梯度分析（text-only vs text-image 专用）

适用前提：每条训练样本只会是两类之一：
- `text-only`
- `text-image`

本版去掉 mixed-batch 相关分析，重点聚焦：
1. 为什么 text-only 与 text-image 曲线相似；
2. 如何在“相似结果”下挖掘更敏感的机制信号。


In [ ]:
import ast, json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style='whitegrid', context='talk')


In [ ]:
LOG_PATH = Path('gradient_logs.jsonl')
OUT = Path('analysis_outputs')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN=None
STEP_MAX=None
STEP_MOD=None
MAX_ROWS=250000
MAX_VEC_ROWS=80000
MAX_PAIR=30000

assert LOG_PATH.exists(), LOG_PATH
print('using', LOG_PATH.resolve())


In [ ]:
def _step_ok(v, lo=None, hi=None, mod=None):
    try: s=int(v)
    except: return False
    if lo is not None and s<lo: return False
    if hi is not None and s>hi: return False
    if mod is not None and mod>1 and s%mod!=0: return False
    return True

def load_subset(path, usecols=None, partition_in=None, max_rows=None):
    usecols=set(usecols) if usecols else None
    rows=[]
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not _step_ok(o['step'], STEP_MIN, STEP_MAX, STEP_MOD):
                continue
            if partition_in is not None and o.get('grad_partition','all') not in partition_in:
                continue
            rows.append(o if usecols is None else {k:o.get(k,None) for k in usecols})
            if max_rows and len(rows)>=max_rows: break
    df=pd.DataFrame(rows)
    if len(df)==0: return df
    for c in ['step','layer_depth','grad_norm','grad_norm_per_token','supervised_token_count','partition_token_ratio','text_token_ratio','image_token_ratio']:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors='coerce')
    if 'grad_partition' in df.columns: df['grad_partition']=df['grad_partition'].fillna('all')
    if 'grad_was_none' in df.columns: df['grad_was_none']=df['grad_was_none'].fillna(False).astype(bool)
    return df


## 1) 基础检查与token-level主趋势

In [ ]:
meta = load_subset(LOG_PATH, usecols=[
    'step','grad_partition','layer_depth','grad_norm','grad_norm_per_token','grad_was_none',
    'partition_token_ratio','text_token_ratio','image_token_ratio','adapter_type','param_name',
    'grad_path','grad'
], max_rows=MAX_ROWS)

if len(meta)==0:
    raise ValueError('No rows loaded; check filters')

if 'grad_norm_per_token' not in meta.columns or meta['grad_norm_per_token'].notna().mean()==0:
    if {'grad_norm','supervised_token_count'}.issubset(meta.columns):
        meta['grad_norm_per_token'] = meta['grad_norm'] / meta['supervised_token_count'].clip(lower=1)

print('rows:', len(meta), 'partitions:', sorted(meta['grad_partition'].dropna().unique().tolist()))


In [ ]:
trend = meta.groupby(['step','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
plt.figure(figsize=(12,5))
sns.lineplot(data=trend, x='step', y='grad_norm_per_token', hue='grad_partition')
plt.title('Mean grad_norm_per_token over steps')
plt.tight_layout(); plt.savefig(OUT/'fig_token_trend_partition.png', dpi=180); plt.show()

layer_mean = meta.groupby(['layer_depth','grad_partition'], as_index=False)['grad_norm_per_token'].mean()
piv = layer_mean.pivot(index='layer_depth', columns='grad_partition', values='grad_norm_per_token').sort_index()
plt.figure(figsize=(10,8)); sns.heatmap(piv, cmap='magma')
plt.title('Layer-depth grouped mean grad_norm_per_token')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_depth_grad_per_token_heatmap.png', dpi=200); plt.show()


## 2) 当两类结果整体相似时：更敏感的差异指标（新增重点）

In [ ]:
# 2.1 分层差异指数（比直接看曲线更敏感）
sub = meta[meta['grad_partition'].isin(['text_only','image_only'])].copy()
key=['step','layer_depth','adapter_type','param_name']
wide = sub.pivot_table(index=key, columns='grad_partition', values='grad_norm_per_token', aggfunc='mean').reset_index()
wide = wide.dropna(subset=['text_only','image_only'])

eps=1e-12
wide['delta'] = wide['image_only'] - wide['text_only']
wide['rel_gap'] = (wide['image_only'] - wide['text_only']) / (wide['image_only'] + wide['text_only'] + eps)
wide['abs_rel_gap'] = wide['rel_gap'].abs()

layer_gap = wide.groupby('layer_depth', as_index=False).agg(
    mean_rel_gap=('rel_gap','mean'),
    mean_abs_rel_gap=('abs_rel_gap','mean'),
    std_rel_gap=('rel_gap','std'),
    n=('rel_gap','count')
)

display(layer_gap.head(12))
layer_gap.to_csv(OUT/'table_layer_depth_gap_stats.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(11,5))
sns.lineplot(data=layer_gap, x='layer_depth', y='mean_abs_rel_gap', marker='o')
plt.title('Layer-wise mean |relative gap| between text_only and image_only')
plt.tight_layout(); plt.savefig(OUT/'fig_layer_abs_rel_gap.png', dpi=180); plt.show()


In [ ]:
# 2.2 稳定性分析：看差异是否持续而非偶然
step_gap = wide.groupby('step', as_index=False)['abs_rel_gap'].mean().sort_values('step')
step_gap['rolling_50'] = step_gap['abs_rel_gap'].rolling(window=50, min_periods=5).mean()

plt.figure(figsize=(12,4))
plt.plot(step_gap['step'], step_gap['abs_rel_gap'], alpha=0.35, label='abs_rel_gap')
plt.plot(step_gap['step'], step_gap['rolling_50'], linewidth=2.0, label='rolling_50')
plt.title('Temporal stability of text/image gap')
plt.legend(); plt.tight_layout(); plt.savefig(OUT/'fig_step_gap_stability.png', dpi=180); plt.show()


In [ ]:
# 2.3 adapter_type分组：很多“整体相似”会在模块层面被打破
if 'adapter_type' in wide.columns:
    mod_gap = wide.groupby(['adapter_type','layer_depth'], as_index=False)['abs_rel_gap'].mean()
    plt.figure(figsize=(12,5))
    sns.lineplot(data=mod_gap, x='layer_depth', y='abs_rel_gap', hue='adapter_type')
    plt.title('Layer-wise |relative gap| by adapter_type')
    plt.tight_layout(); plt.savefig(OUT/'fig_layer_gap_by_adapter_type.png', dpi=180); plt.show()
    mod_gap.to_csv(OUT/'table_layer_gap_by_adapter_type.csv', index=False, encoding='utf-8-sig')


## 3) 向量机制分析（方向冲突+容量）

In [ ]:
def parse_inline_grad(g):
    if g is None:
        return None
    if isinstance(g,(list,tuple,np.ndarray)):
        return np.asarray(g,dtype=np.float32).reshape(-1)
    if isinstance(g,str):
        s=g.strip()
        if not s: return None
        try:
            x=json.loads(s)
            if isinstance(x,list): return np.asarray(x,dtype=np.float32).reshape(-1)
        except: pass
        try:
            x=ast.literal_eval(s)
            if isinstance(x,(list,tuple)): return np.asarray(x,dtype=np.float32).reshape(-1)
        except: pass
    return None

def load_grad_vector(row):
    gp = row.get('grad_path', None)
    if isinstance(gp, str) and gp and Path(gp).exists():
        return np.load(gp).reshape(-1)
    return parse_inline_grad(row.get('grad', None))

vec = load_subset(
    LOG_PATH,
    usecols=['step','param_name','layer_depth','adapter_type','grad_partition','grad_path','grad'],
    partition_in={'text_only','image_only','all'},
    max_rows=MAX_VEC_ROWS,
)
if len(vec)>0:
    vec['vector'] = vec.apply(load_grad_vector, axis=1)
    vec = vec[vec['vector'].notna()].copy()
print('usable vector rows:', len(vec))


In [ ]:
def cosine(a,b,eps=1e-12):
    return float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+eps))

def rank95(X):
    if X.shape[0] < 2: return np.nan
    X = X - X.mean(0, keepdims=True)
    s = np.linalg.svd(X, full_matrices=False, compute_uv=False)
    e = np.cumsum(s*s) / (np.sum(s*s)+1e-12)
    return int(np.searchsorted(e, 0.95)+1)

if len(vec)>0:
    key=['step','param_name','layer_depth','adapter_type']
    t=vec[vec['grad_partition']=='text_only'][key+['vector']].rename(columns={'vector':'tv'})
    i=vec[vec['grad_partition']=='image_only'][key+['vector']].rename(columns={'vector':'iv'})
    pair=t.merge(i,on=key,how='inner').head(MAX_PAIR)

    pair['cos'] = pair.apply(lambda r: cosine(r['tv'][:min(len(r['tv']),len(r['iv']))], r['iv'][:min(len(r['tv']),len(r['iv']))]) if min(len(r['tv']),len(r['iv']))>0 else np.nan, axis=1)
    pair = pair.dropna(subset=['cos'])

    plt.figure(figsize=(9,4)); sns.histplot(pair['cos'], bins=60, kde=True); plt.axvline(0,color='r',ls='--')
    plt.title('Cos(text,image)'); plt.tight_layout(); plt.savefig(OUT/'fig_cos_text_image.png', dpi=180); plt.show()

    layer_cos = pair.groupby('layer_depth', as_index=False)['cos'].agg(['mean','std','count']).reset_index()
    layer_cos.columns = ['layer_depth','cos_mean','cos_std','n']
    layer_cos.to_csv(OUT/'table_layer_depth_cos_stats.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(11,5)); sns.lineplot(data=layer_cos, x='layer_depth', y='cos_mean', marker='o'); plt.axhline(0,color='r',ls='--')
    plt.title('Layer-depth grouped cos(text,image)'); plt.tight_layout(); plt.savefig(OUT/'fig_layer_depth_cos_curve.png', dpi=180); plt.show()

    rec=[]
    for gk,g in vec.groupby(['layer_depth','grad_partition']):
        gg=g.head(180)
        if len(gg)<2: continue
        d=min(len(v) for v in gg['vector'])
        if d<=1: continue
        X=np.stack([v[:d] for v in gg['vector']], axis=0)
        rec.append((gk[0], gk[1], len(gg), d, rank95(X)))
    if len(rec)>0:
        rdf=pd.DataFrame(rec, columns=['layer_depth','grad_partition','n','dim','rank95'])
        rdf.to_csv(OUT/'table_rank95_token_level.csv', index=False, encoding='utf-8-sig')
        plt.figure(figsize=(11,5)); sns.lineplot(data=rdf, x='layer_depth', y='rank95', hue='grad_partition', marker='o')
        plt.title('Rank95 by layer/partition'); plt.tight_layout(); plt.savefig(OUT/'fig_rank95_token_level.png', dpi=180); plt.show()


## 4) 推荐的新实验分析方向（针对“整体相似”）

当 text-only 与 text-image 结果看起来相似，建议优先做：
1. **分层差异指数**（`mean_abs_rel_gap`）而不是只看全局均值曲线；
2. **时间稳定性**（rolling gap），判断差异是否持续；
3. **模块分组**（adapter_type）看是否存在“局部显著差异”；
4. **方向与容量双证据**（layer-wise cos + rank95）定位关键层。
